# Construcción del dataset

Paso 1: juntar en un solo archivo las submissions (`sub.txt`) de salud y energía de todos los zips.

In [1]:
import os
import sys
import zipfile
from glob import glob
from pathlib import Path

import pandas as pd


if "google.colab" in sys.modules:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")

    CANDIDATOS = [
        Path("/content/drive/MyDrive/Colab Notebooks/Diplodatos MAI Grupo 6"),
        Path("/content/drive/MyDrive/Diplodatos MAI Grupo 6"),
    ]
    WORKING_DIR = next((p for p in CANDIDATOS if p.exists()), None)
    if WORKING_DIR is None:
        raise FileNotFoundError(
            "No encuentro la carpeta del grupo. ¿Agregaste el acceso directo "
            "en 'Mi unidad' desde 'Compartido conmigo'?"
        )
else:
    WORKING_DIR = Path.cwd()

In [2]:
SECTORS = {
    "health": (8000, 8099),
    "energy": (1300, 1399),
}

# `glob` lo que hace es listar los que cumplen el patron del argumento, `sorted` los ordena alfabeticamente
ZIPS = sorted(glob(f"{WORKING_DIR}/zips/*.zip"))

DATA_DIR = f"{WORKING_DIR}/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("Primeros 5 archivos zip:", ZIPS[:5])
print(f"Se encontraron {len(ZIPS)} archivos zip")

Primeros 5 archivos zip: ['/home/ebner/FAMAF/Diplodatos/MAI/src/zips/2018q1.zip', '/home/ebner/FAMAF/Diplodatos/MAI/src/zips/2018q2.zip', '/home/ebner/FAMAF/Diplodatos/MAI/src/zips/2018q3.zip', '/home/ebner/FAMAF/Diplodatos/MAI/src/zips/2018q4.zip', '/home/ebner/FAMAF/Diplodatos/MAI/src/zips/2019q1.zip']
Se encontraron 32 archivos zip


Cada fila de `sub.txt` es un documento presentado a la SEC. La columna `sector` dice si la empresa es de salud o de energía, y la columna `zip` dice de qué zip salió la fila.

In [3]:
def read_sub(path: str) -> pd.DataFrame:

    with zipfile.ZipFile(path) as z, z.open("sub.txt") as f:
        submission = pd.read_csv(f, sep="\t", dtype=str)

    sic = pd.to_numeric(submission["sic"], errors="coerce")

    submission["sector"] = None

    for sector, (low, high) in SECTORS.items():
        submission.loc[sic.between(low, high), "sector"] = sector

    # Agregamos una columna con el nombre del zip, para poder identificar de donde viene cada submission
    submission["zip"] = path.split("/")[-1].removesuffix(".zip")
    return submission.dropna(subset=["sector"])


submissions = pd.concat([read_sub(p) for p in ZIPS], ignore_index=True)
submissions.shape

(9918, 38)

Documentos y empresas por sector, y tipos de documento más comunes.

In [4]:
# agrupar la cantidad de documentos y empresas por sector
submissions.groupby("sector").agg(
    submissions=("adsh", "count"),
    companies=("cik", "nunique"),
    # `nunique` cuenta la cantidad de valores distintos (`unique` te da los valores unicos)
)


,submissions,companies
sector,,
energy,6389,379
health,3529,203


In [5]:
# 10-K y 10-Q, con sus enmiendas (/A). docs/decisions.md, sección 3.
submissions = submissions[submissions["form"].isin(["10-K", "10-Q", "10-K/A", "10-Q/A"])]
submissions["form"].value_counts()

form
10-Q      6513
10-K      2184
10-Q/A     161
10-K/A      97
Name: count, dtype: int64

In [6]:
# each submission appears in only one zip
assert submissions["adsh"].is_unique


In [7]:
cols_to_delete = [
    "ein",
    "former",
    "changed",
    "countryba",
    "stprba",
    "cityba",
    "zipba",
    "bas1",
    "bas2",
    "baph",
    "countryma",
    "stprma",
    "cityma",
    "mas1",
    "mas2",
    "zipma",
    "countryinc",
    "stprinc",
    "afs",
    "wksi",
    "accepted",
    "instance",
    "detail",
    "nciks",
    "aciks",
]

submissions.drop(columns=cols_to_delete, inplace=True)

In [8]:
submissions.to_csv(f"{DATA_DIR}/sub.csv", index=False)

In [9]:
submissions.head(5)

,adsh,cik,name,sic,fye,form,period,fy,fp,filed,prevrpt,sector,zip
1,0001185185-18-000404,719209,ISRAMCO INC,1311,1231,10-K,20171231,2017,FY,20180313,0,energy,2018q1
2,0001193125-18-041668,949039,"DIAMOND OFFSHORE DRILLING, INC.",1381,1231,10-K,20171231,2017,FY,20180213,0,energy,2018q1
3,0000318996-18-000007,318996,KEY ENERGY SERVICES INC,1389,1231,10-K,20171231,2017,FY,20180228,0,energy,2018q1
5,0000930413-18-000950,316253,ENZO BIOCHEM INC,8071,0731,10-Q,20180131,2018,Q2,20180312,0,health,2018q1
6,0001214659-18-002003,1364397,RIDGEWOOD ENERGY T FUND LLC,1382,1231,10-K,20171231,2017,FY,20180309,0,energy,2018q1


Paso 2: leer `num.txt` de cada zip y quedarse con los números de los documentos de `submissions`.

`segments` vacío y `coreg` vacío dejan solo los números de la empresa consolidada entera.

In [10]:
NUM_COLS = [
    "adsh",
    "tag",
    "version",
    "ddate",
    "qtrs",
    "uom",
    "segments",
    "coreg",
    "value",
]
US_GAAP_TAXONOMY = [f"us-gaap/{year}" for year in range(2000, 2028)]


def read_num(path: str, adsh: set) -> pd.DataFrame:
    print(f"Reading {path}...")
    with zipfile.ZipFile(path) as z, z.open("num.txt") as f:
        num = pd.read_csv(f, sep="\t", dtype=str, usecols=NUM_COLS)

    num = num[
        num["adsh"].isin(adsh)
        # drops company-defined tags
        & num["version"].isin(US_GAAP_TAXONOMY)
        # drops periods longer than a year
        & num["qtrs"].isin(["0", "1", "2", "3", "4"])
        & num["segments"].isna()
        & num["coreg"].isna()
        # drops non-USD values
        & (num["uom"] == "USD")
    ]
    num = num.dropna(subset=["value"]).drop(columns=["segments", "coreg", "uom"])
    return num


adsh_list = set(submissions["adsh"])
numbers = pd.concat([read_num(p, adsh_list) for p in ZIPS], ignore_index=True)  # ~3 min
numbers.shape

Reading /home/ebner/FAMAF/Diplodatos/MAI/src/zips/2018q1.zip...


Reading /home/ebner/FAMAF/Diplodatos/MAI/src/zips/2018q2.zip...


Reading /home/ebner/FAMAF/Diplodatos/MAI/src/zips/2018q3.zip...


Reading /home/ebner/FAMAF/Diplodatos/MAI/src/zips/2018q4.zip...


Reading /home/ebner/FAMAF/Diplodatos/MAI/src/zips/2019q1.zip...


...

Reading /home/ebner/FAMAF/Diplodatos/MAI/src/zips/2025q1.zip...


Reading /home/ebner/FAMAF/Diplodatos/MAI/src/zips/2025q2.zip...


Reading /home/ebner/FAMAF/Diplodatos/MAI/src/zips/2025q3.zip...


Reading /home/ebner/FAMAF/Diplodatos/MAI/src/zips/2025q4.zip...


(1643181, 6)

In [11]:
numbers.head()

,adsh,tag,version,ddate,qtrs,value
0,0001609253-18-000041,AdjustmentsToAdditionalPaidInCapitalSharebased...,us-gaap/2016,20151231,4,30000000.0000
1,0001564590-18-004217,PaymentsToAcquireFurnitureAndFixtures,us-gaap/2017,20161231,4,96000.0000
2,0001518832-18-000003,AssetsOfDisposalGroupIncludingDiscontinuedOper...,us-gaap/2017,20161231,0,12000000.0000
3,0001537028-18-000029,EarningsPerShareBasicAndDiluted,us-gaap/2017,20161231,4,-0.6700
4,0001652133-18-000009,DeferredIncomeTaxExpenseBenefit,us-gaap/2017,20151231,4,3543000.0000


Paso 3: elegir una versión de cada número.

El mismo número (empresa + concepto + período) aparece en varios documentos: el original, el comparativo de documentos posteriores y a veces una enmienda (`/A`). La regla sale de `docs/rumbo-mai.pdf`: se aceptan las correcciones presentadas hasta el 31/12/2023 (`CUTOFF`) y no las posteriores. Es una foto de la base al 31/12/2023:

- Si el número tiene versiones presentadas hasta `CUTOFF`, gana la **última** de ellas (incluye enmiendas y comparativos).
- Si no, gana la **primera** presentada después de `CUTOFF`. Así los números de 2024 quedan como se publicaron por primera vez.

Se ordena por `filed` y `form`. Si dos documentos tienen el mismo `filed`, `form` desempata: `"10-Q"` va antes que `"10-Q/A"`, así que antes del corte gana la enmienda y después el original.

`filed` queda en el resultado: es la fecha en que se conoció la versión elegida. Ver `docs/decisions.md`, sección 8.

In [12]:
KEY = ["cik", "tag", "ddate", "qtrs"]
CUTOFF = "20231231"  # `filed` is AAAAMMDD, so text order = date order

numbers = numbers.merge(submissions[["adsh", "cik", "form", "filed"]], on="adsh").sort_values(["filed", "form"])
before = numbers[numbers["filed"] <= CUTOFF].drop_duplicates(KEY, keep="last")
after = numbers[numbers["filed"] > CUTOFF].drop_duplicates(KEY, keep="first")

# a version filed before the cutoff wins over any version filed after it
numbers_chosen = pd.concat([before, after]).drop_duplicates(KEY, keep="first")
numbers_chosen.shape

(914961, 9)

In [13]:
# cuántos números elegidos salen de una enmienda
numbers_chosen["form"].value_counts()

form
10-Q      658800
10-K      238800
10-Q/A     11614
10-K/A      5747
Name: count, dtype: int64

In [14]:
numbers_chosen.head()

,adsh,tag,version,ddate,qtrs,value,cik,form,filed
568,0001515971-18-000014,OilAndGasSalesRevenue,us-gaap/2017,20161130,3,332041.0000,1164256,10-Q,20180112
577,0001515971-18-000014,RepaymentsOfLinesOfCredit,us-gaap/2017,20161130,3,30000.0000,1164256,10-Q,20180112
919,0001515971-18-000014,OperatingIncomeLoss,us-gaap/2017,20161130,3,-699458.0000,1164256,10-Q,20180112
1128,0001515971-18-000014,OtherNonoperatingIncomeExpense,us-gaap/2017,20161130,1,-2232899.0000,1164256,10-Q,20180112
1203,0001515971-18-000014,OtherNotesPayableCurrent,us-gaap/2017,20171130,0,9157794.0000,1164256,10-Q,20180112


In [15]:
numbers_chosen.isna().sum()

adsh       0
tag        0
version    0
ddate      0
qtrs       0
value      0
cik        0
form       0
filed      0
dtype: int64

In [16]:
numbers_chosen.to_csv(f"{DATA_DIR}/num.csv", index=False)